# 02 — Stock-market inverse-cubic law

**Goal.** Reproduce the well-known inverse-cubic law of stock-market returns: |daily return| follows a power-law with α ≈ 3.

**ETA.** ~3 minutes.

**Source.** Synthetic Pareto sample seeded with α=3.0, n=20000. (Offline reproducibility — real S&P 500 data via yfinance is included as a commented-out cell.)

**Expected.** α ∈ [2.7, 3.3].

In [ ]:
# 1. Generate / load |daily return| data
import numpy as np

# Synthetic for reproducibility (real S&P 500 below)
rng = np.random.default_rng(42)
alpha_true = 3.0
u = rng.uniform(0.0, 1.0, 20000)
abs_rets = 0.001 * (1.0 - u) ** (-1.0 / (alpha_true - 1.0))
print(f"n = {len(abs_rets)}, max = {abs_rets.max():.4f}, min = {abs_rets.min():.6f}")

# Uncomment to use real S&P 500 (requires yfinance):
# import yfinance as yf, pandas as pd
# df = yf.download('^GSPC', start='1950-01-01', end='2024-12-31', progress=False)
# prices = df['Close'].dropna().values
# rets = np.diff(np.log(prices))
# abs_rets = np.abs(rets[~np.isnan(rets)])
# abs_rets = abs_rets[abs_rets > 0]

In [ ]:
# 2. Fit + bootstrap CI
from soc_pipeline import fit_clauset_powerlaw, bootstrap_ci

fit = fit_clauset_powerlaw(abs_rets, name="|daily return|")
print(f"α = {fit.alpha:.3f} ± {fit.sigma:.3f}")
print(f"xmin = {fit.xmin:.4g}, n_tail = {fit.n_tail}")
print(f"Vuong vs lognormal: R = {fit.vs_lognormal_R}, p = {fit.vs_lognormal_p}")

ci = bootstrap_ci(abs_rets, n_boot=40, seed=42)
print(f"\nbootstrap 95% CI: [{ci.ci_low:.3f}, {ci.ci_high:.3f}]")
print(f"bootstrap median: {ci.alpha_median:.3f}")

In [ ]:
# 3. Log-log CCDF plot
import matplotlib.pyplot as plt
from soc_pipeline import empirical_ccdf

grid, ccdf = empirical_ccdf(abs_rets, n_points=200)
fig, ax = plt.subplots(figsize=(8, 5))
ax.loglog(grid, ccdf, "o", markersize=3, alpha=0.6, label="empirical CCDF")
# Fitted power-law tail
tail_grid = grid[grid >= fit.xmin]
tail_ccdf = (tail_grid / fit.xmin) ** -(fit.alpha - 1) * (abs_rets >= fit.xmin).mean()
ax.loglog(tail_grid, tail_ccdf, "r-", label=f"α = {fit.alpha:.2f}")
ax.axvline(fit.xmin, color="gray", linestyle="--", alpha=0.5)
ax.set_xlabel("|daily return|")
ax.set_ylabel("P(|R| > r)")
ax.set_title("Inverse-cubic law for absolute daily returns")
ax.legend()
ax.grid(True, which="both", alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# 4. Verdict
from soc_pipeline import verdict_from_alpha_band

v = verdict_from_alpha_band(fit.alpha, predicted=(2.8, 3.2), literature=(2.5, 3.5))
print(f"verdict: {v}")
print(f"Paper headline: α ≈ 3 (Mandelbrot 1963, Gopikrishnan et al 1999)")